# Timestomping Detection Tool - Run Detection

This notebook loads the trained LightGBM model and generates predictions on the feature dataset.

## Input

- `data_features.csv` from notebook 02
- `lightgbm_model.pkl` from Phase 3 training

## Process

1. Load trained model
2. Prepare features (same 31 features from training)
3. Generate predictions with confidence scores
4. Flag high-confidence detections (≥70%)

## Output

- `predictions.csv` - All files with confidence scores
- `predictions_with_features.csv` - Predictions with feature values for analysis
- `flagged_files.csv` - High-confidence detections only


## Cell 1: Imports and Setup

In [241]:
# Cell 1: Imports and Setup

import pandas as pd
import numpy as np
import os
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Libraries imported successfully
Pandas version: 2.3.2
NumPy version: 2.3.3


## Cell 2: User Configuration

**EDIT THIS SECTION** if you changed paths in previous notebooks


In [242]:
# Cell 2: User Configuration

# Input file (output from notebook 02)
INPUT_DIR = "/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW"
INPUT_FILE = "data_features.csv"

# Model file (from Phase 3 training)
MODEL_PATH = "/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/lightgbm_model.pkl"

# Output directory
OUTPUT_DIR = "/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LightGBM with Post-Processing (best)"

# Detection threshold (confidence percentage)
CONFIDENCE_THRESHOLD = 70  # Flag files with ≥70% confidence

# Construct paths
input_path = os.path.join(INPUT_DIR, INPUT_FILE)
output_predictions = os.path.join(OUTPUT_DIR, "predictions.csv")
output_predictions_features = os.path.join(OUTPUT_DIR, "predictions_with_features.csv")
output_flagged = os.path.join(OUTPUT_DIR, "flagged_files.csv")

print("Configuration loaded")
print("-" * 80)
print(f"Input file: {input_path}")
print(f"  Exists: {os.path.exists(input_path)}")
print(f"Model file: {MODEL_PATH}")
print(f"  Exists: {os.path.exists(MODEL_PATH)}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}%")
print("-" * 80)
print(f"Output files:")
print(f"  - {output_predictions}")
print(f"  - {output_predictions_features}")
print(f"  - {output_flagged}")

if not os.path.exists(input_path):
    raise FileNotFoundError(f"Feature data not found: {input_path}")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")


Configuration loaded
--------------------------------------------------------------------------------
Input file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/data_features.csv
  Exists: True
Model file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/lightgbm_model.pkl
  Exists: True
Confidence threshold: 70%
--------------------------------------------------------------------------------
Output files:
  - /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LightGBM with Post-Processing (best)/predictions.csv
  - /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LightGBM with Post-Processing (best)/predictions_with_features.csv
  - /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LightGBM with Post-Processing (best)/flagged_files.csv


## Cell 3: Load Feature Dataset


In [243]:
# Cell 3: Load Feature Dataset

print("=" * 80)
print("LOADING FEATURE DATASET")
print("=" * 80)

# Load feature data
df = pd.read_csv(input_path, low_memory=False)

print(f"\nDataset loaded successfully")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")


LOADING FEATURE DATASET

Dataset loaded successfully
  Records: 7,420
  Columns: 64
  Memory usage: 8.82 MB


## Cell 4: Load Trained Model

In [244]:
# Cell 4: Load Trained Model

print("\n" + "=" * 80)
print("LOADING TRAINED MODEL")
print("=" * 80)

# Load model
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

print(f"\nModel loaded successfully")
print(f"  Model type: {type(model).__name__}")
print(f"  Model parameters: {model.get_params()}")

# Get feature names from model (if available)
if hasattr(model, 'feature_name_'):
    model_features = model.feature_name_
    print(f"  Features used in training: {len(model_features)}")
elif hasattr(model, 'n_features_in_'):
    print(f"  Number of features: {model.n_features_in_}")
else:
    print(f"  Feature information not available in model")



LOADING TRAINED MODEL

Model loaded successfully
  Model type: LGBMClassifier
  Model parameters: {'boosting_type': 'gbdt', 'class_weight': 'balanced', 'colsample_bytree': 0.8, 'importance_type': 'split', 'learning_rate': 0.1, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': -1, 'num_leaves': 31, 'objective': None, 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.8, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbose': -1}
  Features used in training: 31


In [245]:
# DIAGNOSTIC: Check model's expected features

print("\n" + "=" * 80)
print("DIAGNOSTIC: MODEL FEATURE EXPECTATIONS")
print("=" * 80)

# Get feature names from model
if hasattr(model, 'feature_name_'):
    expected_features = model.feature_name_
    print(f"\nModel expects {len(expected_features)} features:")
    for i, feat in enumerate(expected_features, 1):
        print(f"  {i:2d}. {feat}")
else:
    print("\nModel feature names not available")
    print(f"Model expects {model.n_features_in_} features (names unknown)")



DIAGNOSTIC: MODEL FEATURE EXPECTATIONS

Model expects 31 features:
   1. cross_artifact_detected
   2. zero_in_nanoseconds_lf
   3. zero_in_nanoseconds_suspicious
   4. zero_in_nanoseconds
   5. time_reversal_event
   6. basic_info_changed
   7. using_another_timestamp
   8. si_timestamp_changed
   9. update_resident_value
  10. creation_time_modified
  11. modified_time_modified
  12. accessed_time_modified
  13. mft_time_modified
  14. timestamp_changed_to_past
  15. multiple_timestamps_changed
  16. same_as_another_file
  17. zero_nano_time_reversal
  18. has_logfile_evidence
  19. has_usnjrnl_evidence
  20. cross_artifact_validation_score
  21. is_executable
  22. is_document
  23. is_archive
  24. is_image
  25. path_depth
  26. filename_length
  27. in_temp_directory
  28. in_system_directory
  29. in_program_files
  30. has_timestamp_data
  31. timestamp_source


## Cell 5: Prepare Features for Prediction

Select same features used during training (exclude raw columns and labels)


In [246]:
# Cell 5: Prepare Features for Prediction (CORRECTED - Match Training Features)

print("\n" + "=" * 80)
print("PREPARING FEATURES FOR PREDICTION")
print("=" * 80)

# Create/rename features to match model's expectations
print("\nCreating features to match training...")

# 1. Rename features to match training names
feature_mapping = {
    'zero_in_nanoseconds_combined': 'zero_in_nanoseconds',
    'modified_creationtime': 'creation_time_modified',
    'modified_modifiedtime': 'modified_time_modified',
    'modified_accessedtime': 'accessed_time_modified',
    'modified_mftmodifiedtime': 'mft_time_modified'
}

for old_name, new_name in feature_mapping.items():
    if old_name in df.columns:
        df[new_name] = df[old_name]
        print(f"  Mapped: {old_name} -> {new_name}")

# 2. Create missing features with default values
missing_features = {
    'cross_artifact_detected': lambda: df['has_logfile_evidence'] & df['has_usnjrnl_evidence'],
    'zero_in_nanoseconds_suspicious': lambda: False,  # Not available in production
    'multiple_timestamps_changed': lambda: (
        df.get('creation_time_modified', False) & 
        df.get('modified_time_modified', False)
    ),
    'same_as_another_file': lambda: False,  # Complex feature, set to False
    'is_image': lambda: df['filename'].fillna('').str.endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp'), na=False),
    'in_temp_directory': lambda: df['full_path'].fillna('').str.contains(r'temp|tmp', case=False, na=False, regex=True),
    'in_system_directory': lambda: df['full_path'].fillna('').str.contains(r'windows|system32|syswow64', case=False, na=False, regex=True),
    'in_program_files': lambda: df['full_path'].fillna('').str.contains(r'program files', case=False, na=False, regex=True),
    'has_timestamp_data': lambda: df['lf_creation_time'].notna() if 'lf_creation_time' in df.columns else False,
    'timestamp_source': lambda: (
        df['has_logfile_evidence'].astype(int) * 2 + 
        df['has_usnjrnl_evidence'].astype(int)
    )  # 0=none, 1=usn, 2=lf, 3=both
}

for feat_name, feat_func in missing_features.items():
    if feat_name not in df.columns:
        df[feat_name] = feat_func()
        print(f"  Created: {feat_name}")

# 3. Select exact features model expects (in correct order)
expected_features = [
    'cross_artifact_detected',
    'zero_in_nanoseconds_lf',
    'zero_in_nanoseconds_suspicious',
    'zero_in_nanoseconds',
    'time_reversal_event',
    'basic_info_changed',
    'using_another_timestamp',
    'si_timestamp_changed',
    'update_resident_value',
    'creation_time_modified',
    'modified_time_modified',
    'accessed_time_modified',
    'mft_time_modified',
    'timestamp_changed_to_past',
    'multiple_timestamps_changed',
    'same_as_another_file',
    'zero_nano_time_reversal',
    'has_logfile_evidence',
    'has_usnjrnl_evidence',
    'cross_artifact_validation_score',
    'is_executable',
    'is_document',
    'is_archive',
    'is_image',
    'path_depth',
    'filename_length',
    'in_temp_directory',
    'in_system_directory',
    'in_program_files',
    'has_timestamp_data',
    'timestamp_source'
]

# Build feature matrix with exact features
X = df[expected_features].copy()

# Handle missing values
X = X.fillna(0)

# Convert boolean to int
for col in X.columns:
    if X[col].dtype == 'bool':
        X[col] = X[col].astype(int)

print(f"\nFeatures prepared:")
print(f"  Features expected by model: {len(expected_features)}")
print(f"  Features provided: {len(X.columns)}")
print(f"  Feature matrix shape: {X.shape}")

# Verify all features present
missing = set(expected_features) - set(X.columns)
if missing:
    print(f"\nWARNING: Missing features: {missing}")
else:
    print(f"\nAll required features present!")



PREPARING FEATURES FOR PREDICTION

Creating features to match training...
  Mapped: zero_in_nanoseconds_combined -> zero_in_nanoseconds
  Mapped: modified_creationtime -> creation_time_modified
  Mapped: modified_modifiedtime -> modified_time_modified
  Mapped: modified_accessedtime -> accessed_time_modified
  Mapped: modified_mftmodifiedtime -> mft_time_modified
  Created: cross_artifact_detected
  Created: multiple_timestamps_changed
  Created: same_as_another_file
  Created: is_image
  Created: in_temp_directory


  Created: in_system_directory
  Created: in_program_files
  Created: has_timestamp_data
  Created: timestamp_source

Features prepared:
  Features expected by model: 31
  Features provided: 31
  Feature matrix shape: (7420, 31)

All required features present!


## Cell 6: Generate Predictions


In [247]:
# Cell 6: Generate Predictions

print("\n" + "=" * 80)
print("GENERATING PREDICTIONS")
print("=" * 80)

# Generate probability predictions
print("\nRunning model predictions...")
y_pred_proba = model.predict_proba(X)

# Extract probability for positive class (timestomped)
if y_pred_proba.ndim == 2:
    # Binary classification - take probability of class 1
    prediction_proba = y_pred_proba[:, 1]
else:
    # Single probability output
    prediction_proba = y_pred_proba

# Convert to confidence percentage
confidence_pct = prediction_proba * 100

# Binary predictions (threshold at confidence_threshold)
is_flagged = confidence_pct >= CONFIDENCE_THRESHOLD

print(f"\nPredictions generated:")
print(f"  Total files: {len(df):,}")
print(f"  Flagged (≥{CONFIDENCE_THRESHOLD}% confidence): {is_flagged.sum():,}")
print(f"  Not flagged (<{CONFIDENCE_THRESHOLD}% confidence): {(~is_flagged).sum():,}")

# Add predictions to dataframe
df['prediction_proba'] = prediction_proba
df['confidence_pct'] = confidence_pct
df['is_flagged'] = is_flagged

# Display confidence distribution
print(f"\nConfidence distribution:")
print(f"  Minimum: {confidence_pct.min():.2f}%")
print(f"  Maximum: {confidence_pct.max():.2f}%")
print(f"  Mean: {confidence_pct.mean():.2f}%")
print(f"  Median: {np.median(confidence_pct):.2f}%")  # FIXED: Use np.median()

# Confidence bins
bins = [0, 50, 70, 80, 90, 100]
labels = ['<50%', '50-70%', '70-80%', '80-90%', '90-100%']
df['confidence_bin'] = pd.cut(confidence_pct, bins=bins, labels=labels, include_lowest=True)

print(f"\nConfidence bins:")
print(df['confidence_bin'].value_counts().sort_index())



GENERATING PREDICTIONS

Running model predictions...

Predictions generated:
  Total files: 7,420
  Flagged (≥70% confidence): 24
  Not flagged (<70% confidence): 7,396

Confidence distribution:
  Minimum: 0.00%
  Maximum: 99.84%
  Mean: 1.72%
  Median: 0.00%

Confidence bins:
confidence_bin
<50%       7396
50-70%        0
70-80%        0
80-90%        0
90-100%      24
Name: count, dtype: int64


## Cell 6b: Apply Minimal Post-Processing Filter


In [248]:
print("\n" + "=" * 80)
print("APPLYING MINIMAL POST-PROCESSING FILTER")
print("=" * 80)

def apply_minimal_filtering(row):
    """
    Filter well-documented benign system operations.
    
    Returns:
    - KEEP: File should be reviewed by investigator
    - FILTER: Well-documented benign system operation
    """
    
    filename = str(row['filename']).lower()
    full_path = str(row['full_path']).lower()
    
    # === RULE 1: Windows Appraiser Files ===
    # Windows Update compatibility assessment files
    # Reference: Microsoft Windows Update documentation
    if filename in ['appraiser.sdb', 'appraiser_data.ini', 'appraiser_telemetryrunlist.xml']:
        if 'windows\\appcompat\\appraiser' in full_path:
            return 'FILTER'
    
    # === RULE 2: Installer Temp Files ===
    # Pattern: Set[HEX].tmp in AppData\Local\Temp
    # Common installer temporary file pattern
    if filename.startswith('set') and filename.endswith('.tmp'):
        if 'appdata\\local\\temp' in full_path:
            return 'FILTER'
    
    # === RULE 3: Windows Update Temp Files ===
    # Pattern: UDD[HEX].tmp in Windows\Temp
    # Windows Update Deployment temporary files
    if filename.startswith('udd') and filename.endswith('.tmp'):
        if 'windows\\temp' in full_path:
            return 'FILTER'
    
    return 'KEEP'

# Apply filtering to ALL files
df['filter_decision'] = df.apply(apply_minimal_filtering, axis=1)

# Calculate filtering statistics
flagged_before = is_flagged.sum()
filtered_files = df[is_flagged & (df['filter_decision'] == 'FILTER')]
kept_files = df[is_flagged & (df['filter_decision'] == 'KEEP')]

print(f"\nPost-Processing Results:")
print(f"  ML model flagged: {flagged_before:,} files")
print(f"  Filtered (benign): {len(filtered_files):,} files")
print(f"  Final flagged (for review): {len(kept_files):,} files")

if len(filtered_files) > 0:
    print(f"\n{'='*80}")
    print(f"FILTERED FILES (Well-Documented Benign Operations)")
    print(f"{'='*80}")
    
    for idx, row in filtered_files.iterrows():
        print(f"\n  ✓ {row['filename']}")
        print(f"    Path: {row['full_path']}")
        print(f"    Confidence: {row['confidence_pct']:.1f}%")
        print(f"    Reason: ", end="")
        
        fname = row['filename'].lower()
        if 'appraiser' in fname:
            print("Windows Update compatibility assessment file")
        elif fname.startswith('set') and fname.endswith('.tmp'):
            print("Software installer temporary file")
        elif fname.startswith('udd') and fname.endswith('.tmp'):
            print("Windows Update Deployment temporary file")

# *** CRITICAL: Update is_flagged to only include KEEP files ***
is_flagged_final = is_flagged & (df['filter_decision'] == 'KEEP')
df['is_flagged'] = is_flagged_final  # Overwrite is_flagged for downstream cells

print(f"\n{'='*80}")
print(f"✓ Minimal post-processing applied")
print(f"  Files requiring investigator review: {is_flagged_final.sum():,}")
print(f"{'='*80}")



APPLYING MINIMAL POST-PROCESSING FILTER

Post-Processing Results:
  ML model flagged: 24 files
  Filtered (benign): 7 files
  Final flagged (for review): 17 files

FILTERED FILES (Well-Documented Benign Operations)

  ✓ appraiser.sdb
    Path: \Windows\appcompat\appraiser\AltData\appraiser.sdb
    Confidence: 99.8%
    Reason: Windows Update compatibility assessment file

  ✓ Appraiser_Data.ini
    Path: \Windows\appcompat\appraiser\AltData\Appraiser_Data.ini
    Confidence: 99.7%
    Reason: Windows Update compatibility assessment file

  ✓ Appraiser_TelemetryRunList.xml
    Path: \Windows\appcompat\appraiser\AltData\Appraiser_TelemetryRunList.xml
    Confidence: 99.8%
    Reason: Windows Update compatibility assessment file

  ✓ Set9A28.tmp
    Path: \Users\jcloudy\AppData\Local\Temp\Set9A28.tmp
    Confidence: 99.8%
    Reason: Software installer temporary file

  ✓ Set9A38.tmp
    Path: \Users\jcloudy\AppData\Local\Temp\Set9A38.tmp
    Confidence: 99.8%
    Reason: Software instal

## Cell 7: Analyze Flagged Files


In [249]:
# Cell 7: Analyze Flagged Files

print("\n" + "=" * 80)
print("ANALYZING FLAGGED FILES")
print("=" * 80)

# Get flagged files
flagged_df = df[df['is_flagged']].copy()

print(f"\nFlagged files: {len(flagged_df):,}")

if len(flagged_df) > 0:
    # Sort by confidence (highest first)
    flagged_df = flagged_df.sort_values('confidence_pct', ascending=False)
    
    # Confidence stats for flagged files
    print(f"\nConfidence statistics (flagged files only):")
    print(f"  Minimum: {flagged_df['confidence_pct'].min():.2f}%")
    print(f"  Maximum: {flagged_df['confidence_pct'].max():.2f}%")
    print(f"  Mean: {flagged_df['confidence_pct'].mean():.2f}%")
    print(f"  Median: {flagged_df['confidence_pct'].median():.2f}%")
    
    # Feature correlation with flagged files
    print(f"\nTop features in flagged files:")
    
    feature_presence = {}
    for feat in feature_cols:
        if feat in flagged_df.columns and flagged_df[feat].dtype in ['bool', 'int64', 'float64']:
            if flagged_df[feat].dtype == 'bool':
                count = flagged_df[feat].sum()
            else:
                count = (flagged_df[feat] > 0).sum()
            
            pct = (count / len(flagged_df)) * 100
            feature_presence[feat] = (count, pct)
    
    # Sort by percentage
    top_features = sorted(feature_presence.items(), key=lambda x: x[1][1], reverse=True)[:10]
    
    for feat, (count, pct) in top_features:
        print(f"  {feat:40s}: {count:4,}/{len(flagged_df):,} ({pct:5.1f}%)")
    
    # Display top 10 flagged files
    print(f"\nTop 10 flagged files (by confidence):")
    display_cols = ['filename', 'confidence_pct', 'zero_in_nanoseconds_combined', 
                    'time_reversal_event', 'basic_info_changed']
    available_display = [col for col in display_cols if col in flagged_df.columns]
    print(flagged_df[available_display].head(10).to_string(index=False))
else:
    print("\nNo files flagged - model may be too conservative or data has no timestomping signals")



ANALYZING FLAGGED FILES

Flagged files: 17

Confidence statistics (flagged files only):
  Minimum: 98.02%
  Maximum: 99.80%
  Mean: 98.86%
  Median: 98.92%

Top features in flagged files:
  has_logfile_evidence                    :   17/17 (100.0%)
  has_usnjrnl_evidence                    :   17/17 (100.0%)
  time_reversal_event                     :   17/17 (100.0%)
  modified_modifiedtime                   :   17/17 (100.0%)
  basic_info_changed                      :   17/17 (100.0%)
  cross_artifact_validation_score         :   17/17 (100.0%)
  path_depth                              :   17/17 (100.0%)
  filename_length                         :   17/17 (100.0%)
  has_file_extension                      :   17/17 (100.0%)
  zero_in_nanoseconds_lf                  :   16/17 ( 94.1%)

Top 10 flagged files (by confidence):
       filename  confidence_pct  zero_in_nanoseconds_combined  time_reversal_event  basic_info_changed
   snapshot.etl       99.804303                         Fal

## Cell 8: Feature Importance Analysis


In [250]:
# Cell 8: Feature Importance Analysis

print("\n" + "=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

# Get feature importance from model
if hasattr(model, 'feature_importances_'):
    importance = model.feature_importances_
    
    # Get feature names from Cell 5
    feature_names = expected_features  # FIXED: Use expected_features instead of feature_cols
    
    # Create importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    })
    
    # Sort by importance
    importance_df = importance_df.sort_values('importance', ascending=False)
    
    print(f"\nTop 15 most important features:")
    print(importance_df.head(15).to_string(index=False))
    
    # Validate critical features are important
    print(f"\nValidating critical features:")
    critical_features = ['zero_in_nanoseconds', 'zero_in_nanoseconds_lf', 
                         'time_reversal_event', 'cross_artifact_validation_score',
                         'basic_info_changed', 'cross_artifact_detected']
    
    for feat in critical_features:
        if feat in importance_df['feature'].values:
            rank = importance_df[importance_df['feature'] == feat].index[0] + 1
            imp = importance_df[importance_df['feature'] == feat]['importance'].values[0]
            print(f"  {feat:40s}: Rank #{rank:2d}, Importance: {imp:.4f}")
        else:
            print(f"  {feat:40s}: Not found in features")
else:
    print("\nFeature importance not available for this model type")



FEATURE IMPORTANCE ANALYSIS

Top 15 most important features:
                        feature  importance
                filename_length        1030
                     path_depth         494
              in_temp_directory         187
            in_system_directory         142
               timestamp_source         136
    multiple_timestamps_changed         113
        cross_artifact_detected         108
                  is_executable          97
               in_program_files          94
             basic_info_changed          91
              mft_time_modified          83
cross_artifact_validation_score          78
         zero_in_nanoseconds_lf          63
         creation_time_modified          44
            time_reversal_event          41

Validating critical features:
  zero_in_nanoseconds                     : Rank # 4, Importance: 6.0000
  zero_in_nanoseconds_lf                  : Rank # 2, Importance: 63.0000
  time_reversal_event                     : Rank # 5, Im

## Cell 9: Save Output Files


In [251]:
# Cell 9: Save Output Files (ENHANCED - Includes Forensic Investigation Details)

import os

print("\n" + "="*80)
print("SAVING OUTPUT FILES WITH FORENSIC DETAILS")
print("="*80)

# *** IMPORTANT: Recreate flagged_df with only KEEP files ***
# This ensures we only save files that passed post-processing
flagged_df = df[df['is_flagged']].copy()  # is_flagged was updated in Cell 6b

print(f"\nFinal flagged files to save: {len(flagged_df):,}")
print(f"  (After minimal post-processing filter)")


# Define comprehensive output columns for forensic investigation
flagged_output_cols = [
    # ===== FILE IDENTIFICATION =====
    'filename', 
    'full_path',
    
    # ===== FORENSIC IDENTIFIERS (for cross-reference with original artifacts) =====
    'lf_lsn',      # LogFile LSN - use to look up in LogFile CSV
    'usn_usn',     # UsnJrnl USN - use to look up in UsnJrnl CSV
    
    # ===== DETECTION CONFIDENCE =====
    'confidence_pct',
    
    # ===== EVENT TIMELINE =====
    'usn_event_time',   # When the timestomping event occurred
    'lf_event_time',    # LogFile event timestamp (if available)
    
    # ===== MANIPULATION DETAILS (Critical!) =====
    'lf_event',         # Event type: "Time Reversal Event", "Update Resident Value", etc.
    'lf_detail',        # SHOWS EXACT MANIPULATION: "ModifiedTime: FROM -> TO (Zero in 100-nanoseconds)"
    'usn_event_info',   # UsnJrnl event info: "Basic_Info_Changed / File_Closed"
    
    # ===== FORENSIC INDICATORS =====
    'zero_in_nanoseconds',           # Combined zero nanoseconds detection
    'zero_in_nanoseconds_lf',        # Zero nanoseconds from LogFile Detail
    'time_reversal_event',           # Timestamp changed to PAST
    'basic_info_changed',            # File metadata/timestamps modified
    'cross_artifact_validation_score', # 1.0 = both sources agree (high confidence)
    'cross_artifact_detected',       # Both LogFile AND UsnJrnl detected
    'has_logfile_evidence',          # Evidence in LogFile
    'has_usnjrnl_evidence',          # Evidence in UsnJrnl
    
    # ===== TIMESTAMP VALUES =====
    'lf_creation_time',     # File CreationTime
    'lf_modified_time',     # File ModifiedTime
    'lf_accessed_time',     # File AccessedTime
    'lf_mft_modified_time', # MFT ModifiedTime
    
    # ===== ADDITIONAL INDICATORS =====
    'timestamp_changed_to_past',  # Timestamp backdated
    'using_another_timestamp',    # Using timestamp from another file
    'modified_creationtime',      # CreationTime was modified
    'modified_modifiedtime',      # ModifiedTime was modified
    'modified_accessedtime',      # AccessedTime was modified
    'zero_nano_time_reversal',    # Time Reversal with zero nanoseconds
    
    # ===== FILE CHARACTERISTICS =====
    'path_depth',        # Directory depth (suspicious if in unusual location)
    'filename_length',   # Filename length
    'is_executable',     # .exe, .dll, .sys files
    'is_document',       # Office documents
    'is_archive',        # .zip, .rar files
    'is_image'           # Image files
]

# Filter to only columns that exist in the dataframe
available_cols = [col for col in flagged_output_cols if col in df.columns]

print(f"Total columns defined: {len(flagged_output_cols)}")
print(f"Columns available in data: {len(available_cols)}")

# ===== CONFIDENCE THRESHOLD =====
CONFIDENCE_THRESHOLD = 70  # Flag files with >= 70% confidence

# Create flagged label based on confidence threshold
df['flagged'] = df['confidence_pct'] >= CONFIDENCE_THRESHOLD

print(f"\nConfidence threshold: {CONFIDENCE_THRESHOLD}%")
print(f"Files flagged: {df['flagged'].sum():,} out of {len(df):,}")

# ===== Save Flagged Files (High Confidence) =====
flagged_df = df[df['flagged']][available_cols].copy()
flagged_df = flagged_df.sort_values('confidence_pct', ascending=False)

flagged_output_path = f'{OUTPUT_DIR}/flagged_files.csv'
flagged_df.to_csv(flagged_output_path, index=False)

print(f"\n✓ Saved flagged files: {len(flagged_df):,} files")
print(f"  Columns included: {len(available_cols)}")
print(f"  Output: {flagged_output_path}")
print(f"  File size: {os.path.getsize(flagged_output_path) / 1024:.2f} KB")

# ===== Save All Predictions (with probabilities) =====
all_predictions_cols = ['filename', 'full_path', 'lf_lsn', 'usn_usn', 'confidence_pct', 'flagged']
all_pred_available = [col for col in all_predictions_cols if col in df.columns]

predictions_output_path = f'{OUTPUT_DIR}/predictions.csv'
df[all_pred_available].to_csv(predictions_output_path, index=False)

print(f"\n✓ Saved all predictions: {len(df):,} files")
print(f"  Output: {predictions_output_path}")

# ===== Save Detailed Analysis (flagged files with all features) =====
detailed_output_path = f'{OUTPUT_DIR}/predictions_with_features.csv'
predictions_with_features_df = df[df['flagged']].copy()
predictions_with_features_df.to_csv(detailed_output_path, index=False)

print(f"\n✓ Saved detailed predictions: {len(predictions_with_features_df):,} files")
print(f"  All {len(predictions_with_features_df.columns)} features included")
print(f"  Output: {detailed_output_path}")

# ===== Display Sample Forensic Report =====
print("\n" + "="*80)
print("SAMPLE FORENSIC INVESTIGATION REPORT")
print("="*80)

if len(flagged_df) > 0:
    sample = flagged_df.iloc[0]
    
    print(f"\nFILE: {sample.get('filename', 'N/A')}")
    print(f"LOCATION: {sample.get('full_path', 'N/A')}")
    print(f"CONFIDENCE: {sample.get('confidence_pct', 0):.2f}%")
    
    print("\n--- EVENT TIMELINE ---")
    if pd.notna(sample.get('usn_event_time')):
        print(f"When: {sample['usn_event_time']}")
    
    print("\n--- MANIPULATION DETECTED ---")
    if pd.notna(sample.get('lf_event')):
        print(f"Type: {sample['lf_event']}")
    if pd.notna(sample.get('lf_detail')):
        detail = str(sample['lf_detail'])
        if len(detail) > 150:
            print(f"Detail: {detail[:150]}...")
        else:
            print(f"Detail: {detail}")
    if pd.notna(sample.get('usn_event_info')):
        print(f"UsnJrnl Event: {sample['usn_event_info']}")
    
    print("\n--- FORENSIC EVIDENCE ---")
    indicators = []
    if sample.get('zero_in_nanoseconds', False):
        indicators.append("✓ Zero nanoseconds (SetFileTime API signature)")
    if sample.get('time_reversal_event', False):
        indicators.append("✓ Time Reversal (timestamp changed to PAST)")
    if sample.get('basic_info_changed', False):
        indicators.append("✓ Basic_Info_Changed in UsnJrnl")
    if sample.get('cross_artifact_validation_score', 0) == 1.0:
        indicators.append("✓ Cross-artifact validation (both LogFile + UsnJrnl agree)")
    
    for indicator in indicators:
        print(f"  {indicator}")
    
    print("\n--- CROSS-REFERENCE IDENTIFIERS ---")
    if pd.notna(sample.get('lf_lsn')):
        print(f"LogFile LSN: {sample['lf_lsn']:.0f}")
    if pd.notna(sample.get('usn_usn')):
        print(f"UsnJrnl USN: {sample['usn_usn']:.0f}")
    
    print("\n--- INVESTIGATION ACTION ---")
    print("  → Review file for malicious activity")
    print("  → Verify timestamp manipulation manually")
    print("  → Correlate with other system events at same time")
    print("  → Check if file is part of attack campaign")

else:
    print("\nNo files flagged above confidence threshold.")

print("\n" + "="*80)
print("OUTPUT SUMMARY")
print("="*80)
print(f"1. flagged_files.csv       - {len(flagged_df):,} high-confidence detections with forensic details")
print(f"2. predictions.csv         - {len(df):,} files with confidence scores")
print(f"3. predictions_with_features.csv - {len(predictions_with_features_df):,} flagged files with all features")
print("\n✓ All output files saved successfully!")



SAVING OUTPUT FILES WITH FORENSIC DETAILS

Final flagged files to save: 17
  (After minimal post-processing filter)
Total columns defined: 34
Columns available in data: 33

Confidence threshold: 70%
Files flagged: 24 out of 7,420

✓ Saved flagged files: 24 files
  Columns included: 33
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LightGBM with Post-Processing (best)/flagged_files.csv
  File size: 9.22 KB

✓ Saved all predictions: 7,420 files
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LightGBM with Post-Processing (best)/predictions.csv

✓ Saved detailed predictions: 24 files
  All 84 features included
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LightGBM with Post-Processing (best)/predictions_with_features.csv

SAMPLE FORENSIC INVESTIGATION REPORT

FILE: appraiser.sdb
LOCATION: \Windows\appcompat\appraiser\AltData\appraiser.sdb
CONFIDENCE: 99.8